# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
!pip -q install duckdb huggingface_hub pandas pyarrow

import duckdb
import pandas as pd
from huggingface_hub import login
from google.colab import userdata

token = userdata.get("HF_TOKEN")
login(token)

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{token}'
);
""")

print("✅ Setup completed successfully!")

✅ Setup completed successfully!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [10]:
print("""
Unit of Analysis

One row represents the daily performance of one content item
for one client on one report date.

Table Used:
fact_content_daily_performance (March 2026 partition)

Time Window:
March 2026 (month = '2026-03')

Prediction Goal:
Rank content pages based on their refresh priority.

Excluded:
Future information and label-derived columns are excluded.
""")


Unit of Analysis

One row represents the daily performance of one content item
for one client on one report date.

Table Used:
fact_content_daily_performance (March 2026 partition)

Time Window:
March 2026 (month = '2026-03')

Prediction Goal:
Rank content pages based on their refresh priority.

Excluded:
Future information and label-derived columns are excluded.



In [11]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet';
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [19]:
print("""
Fields Classification

FEATURES
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

Reason:
These values are available before making a content refresh decision, so they are safe to use as model features.

LABEL / PROXY
Refresh Priority Ranking
The objective is to rank content pages that should be refreshed based on historical search and engagement performance.


CONTEXT
- client_hash_id
- content_hash_id
- report_date
- month

Reason:
These fields identify the content item and date but are not used for prediction.

EXCLUDED
• ga4_data_available
• gsc_data_available
• trend_direction
• trend_pct
• is_declining_label
• Future information

Reason:
These columns are label-derived or contain future information and would cause data leakage if used as model features.
""")


Fields Classification

FEATURES
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

Reason:
These values are available before making a content refresh decision, so they are safe to use as model features.

LABEL / PROXY
Refresh Priority Ranking
The objective is to rank content pages that should be refreshed based on historical search and engagement performance.


CONTEXT
- client_hash_id
- content_hash_id
- report_date
- month

Reason:
These fields identify the content item and date but are not used for prediction.

EXCLUDED
• ga4_data_available
• gsc_data_available
• trend_direction
• trend_pct
• is_declining_label
• Future information

Reason:
These columns are label-derived or contain future information and would cause data leakage if used as model features.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt


In [14]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet';
"""

con.sql(query).df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [15]:
query = """
SELECT
    COUNT(*) AS available_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
WHERE ga4_data_available IS TRUE;
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


### FIVE FEATURE FRAME

In [21]:
okiequery = """
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
WHERE ga4_data_available IS TRUE
LIMIT 5;
"""

con.sql(query).df()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,0,0,NaN,1,1
1,0,0,NaN,1,1
2,0,0,NaN,1,1
3,0,0,NaN,1,1
4,0,0,NaN,1,1


### Why these features are safe

- gsc_impressions — Known before the refresh decision because they are historical search impressions.
- gsc_clicks — Historical clicks already observed before prediction.
- gsc_avg_position — Search ranking known before the decision point.
- ga4_pageviews — Historical page views available before prediction.
- ga4_sessions — Historical user sessions available before prediction.

### Leakage Demonstration

In [22]:
print("""
Leakage Demonstration

A label-derived or future-information column can make model performance appear unrealistically high.

Examples of leakage:
- trend_pct
- trend_direction
- is_declining_label

These columns encode information about the prediction target or future outcomes.

Therefore, they must never be used as model features and are intentionally excluded from this feature set.
""")


Leakage Demonstration

A label-derived or future-information column can make model performance appear unrealistically high.

Examples of leakage:
- trend_pct
- trend_direction
- is_declining_label

These columns encode information about the prediction target or future outcomes.

Therefore, they must never be used as model features and are intentionally excluded from this feature set.



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [20]:
print("""
Data Limitations

1. Different clients have different amounts of historical data, so comparisons across clients may not always be fair.

2. Some rows do not have Google Analytics (GA4) data available. These rows must be filtered using the availability flag.

3. This dataset contains observational data, so it can identify patterns but cannot prove cause-and-effect relationships.

4. IDs such as client_hash_id and content_hash_id are used only for identification and should never be model features.

5. The sample dataset contains only the final month (June 2026), while model development should use a mid-panel month such as March 2026 to avoid data leakage.

6. Only records with ga4_data_available = TRUE are used for feature construction, reducing the amount of usable data for some clients.
""")


Data Limitations

1. Different clients have different amounts of historical data, so comparisons across clients may not always be fair.

2. Some rows do not have Google Analytics (GA4) data available. These rows must be filtered using the availability flag.

3. This dataset contains observational data, so it can identify patterns but cannot prove cause-and-effect relationships.

4. IDs such as client_hash_id and content_hash_id are used only for identification and should never be model features.

5. The sample dataset contains only the final month (June 2026), while model development should use a mid-panel month such as March 2026 to avoid data leakage.

6. Only records with ga4_data_available = TRUE are used for feature construction, reducing the amount of usable data for some clients.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.